# Pittsburgh Traffic Flow, Part 3: Modeled Road-Level Intensity & Top Commute Routes

**Newsletter series: Geospatial Data & Public Transportation — Project 2 of 5 (EDA, notebook 3)**

Two things this notebook builds:

1. **A green/yellow/red road-level intensity map.** Real per-street vehicle counts would need sensor data (the PennDOT license we already decided to avoid). Instead, we use **traffic assignment**: route every commuting flow over the real road network and add up how many commuters cross each street segment. This is a *modeled estimate* of where flow concentrates -- a standard first-pass transportation-planning technique -- not a measurement. Worth sitting with that distinction; it's also a direct, simplified preview of what Project 4's simulation will do more rigorously.
2. **The top 3 heaviest commute routes**, drawn as actual routed paths over real streets (not straight desire lines).

**A note on last notebook's bedroom-community gap:** none of Pittsburgh's tracts showed up as clear bedroom communities. That's a real artifact of the filter, not the city: we only counted a tract's outbound workers if their *workplace* was also inside Allegheny County. Anyone commuting out of county (or out of state, or working remotely) never got counted as an "outflow" at all -- so every tract's outflow was structurally undercounted, especially ones near the county line. Worth knowing about, not worth fixing for this project's purposes.

**One infrastructure change:** Notebooks 1-2 pulled a road network clipped to Pittsburgh's city limits. Several of the heaviest commute flows originate in suburbs outside that boundary (Wexford, Franklin Park, etc.), so routing needs the full county network this time. That means a bigger download than previous notebooks -- expect this to take noticeably longer to run.

## Setup

In [ ]:
!pip install osmnx geopandas contextily pygris -q


In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import contextily as cx
import requests
import io
from shapely.geometry import LineString
from collections import defaultdict

ox.settings.log_console = False

PGH_BLACK = "#1a1a1a"
INTENSITY_COLORS = {"Low": "#2e8b57", "Medium": "#f2b134", "High": "#c1440e"}
ROUTE_COLORS = ["#1f6fb2", "#7b2d8b", "#1a1a1a"]


## Step 1: Pull the county-wide road network

This covers all of Allegheny County, so routes can reach suburban commute origins. Expect this cell to take a few minutes.

In [ ]:
place = "Allegheny County, Pennsylvania, USA"

G = ox.graph_from_place(place, network_type="drive")
print(f"Nodes: {len(G.nodes):,}   Edges: {len(G.edges):,}")

G_proj = ox.project_graph(G)
G_crs = G_proj.graph["crs"]
nodes, edges = ox.graph_to_gdfs(G_proj)


## Step 2: Road hierarchy + approximate travel time

Same classification as Notebook 1, now used for something new: assigning a rough free-flow speed per class so shortest-path routing reflects realistic travel time, not just raw distance. These speed assumptions are illustrative planning-level defaults, not measured limits for any specific street.

In [ ]:
HIERARCHY_MAP = {
    "motorway": "Highway", "motorway_link": "Highway",
    "trunk": "Highway", "trunk_link": "Highway",
    "primary": "Primary", "primary_link": "Primary",
    "secondary": "Secondary", "secondary_link": "Secondary",
    "tertiary": "Tertiary", "tertiary_link": "Tertiary",
}

def classify(row):
    hw = row.get("highway")
    if isinstance(hw, list):
        hw = hw[0]
    return HIERARCHY_MAP.get(hw, "Local")

SPEED_MPH = {"Highway": 55, "Primary": 35, "Secondary": 30, "Tertiary": 25, "Local": 20}

edges["road_class"] = edges.apply(classify, axis=1)
edges["speed_mps"] = edges["road_class"].map(SPEED_MPH) * 0.44704  # mph -> m/s
edges["travel_time"] = edges["length"] / edges["speed_mps"]

# Push travel_time back onto the graph so we can route by it
tt_lookup = edges["travel_time"].to_dict()  # keyed by (u, v, key)
nx.set_edge_attributes(G_proj, tt_lookup, "travel_time")


## Step 3: Pull commuting flows again (tract-pair level)

Same LODES source as Notebook 2. We take more pairs this time (not just the top 40) so the assignment map has enough routed flow to look like a real network, not a handful of isolated corridors.

In [ ]:
LODES_YEAR = 2022  # check https://lehd.ces.census.gov/data/#lodes for newer years
LODES_URL = f"https://lehd.ces.census.gov/data/lodes/LODES8/pa/od/pa_od_main_JT00_{LODES_YEAR}.csv.gz"

resp = requests.get(LODES_URL)
resp.raise_for_status()
od = pd.read_csv(io.BytesIO(resp.content), compression="gzip", dtype={"w_geocode": str, "h_geocode": str})

ALLEGHENY_FIPS = "42003"
od_allegheny = od[od["w_geocode"].str[:5] == ALLEGHENY_FIPS].copy()
od_allegheny["w_tract"] = od_allegheny["w_geocode"].str[:11]
od_allegheny["h_tract"] = od_allegheny["h_geocode"].str[:11]

pair_flows = od_allegheny.groupby(["h_tract", "w_tract"])["S000"].sum().reset_index()
pair_flows = pair_flows.rename(columns={"S000": "flow"})
pair_flows = pair_flows[pair_flows["h_tract"] != pair_flows["w_tract"]]

print(f"Tract-pair flows: {len(pair_flows):,}")


In [ ]:
from pygris import tracts

pa_tracts = tracts(state="PA", county="Allegheny", cb=True, year=2022).to_crs(G_crs)
pa_tracts["centroid"] = pa_tracts.geometry.centroid
centroids = pa_tracts.set_index("GEOID")["centroid"]

# Keep only pairs where both ends have a known tract centroid within the county
pair_flows = pair_flows[pair_flows["h_tract"].isin(centroids.index) & pair_flows["w_tract"].isin(centroids.index)]

TOP_N_ASSIGN = 150  # more pairs = richer map, but longer routing time
assign_pairs = pair_flows.nlargest(TOP_N_ASSIGN, "flow").copy()
assign_pairs["h_pt"] = assign_pairs["h_tract"].map(centroids)
assign_pairs["w_pt"] = assign_pairs["w_tract"].map(centroids)

print(f"Pairs selected for assignment: {len(assign_pairs)}")


## Step 4: Route each flow over the real network

For each pair, we find the nearest network node to each tract centroid, then compute the shortest (fastest, by our estimated travel time) path between them -- and add that pair's commuter count onto every street segment along the way. This is exactly what real-world traffic assignment models do, just at a much simpler scale.

This is the slow cell -- routing 150 shortest paths over a county-sized network.

In [ ]:
h_x = assign_pairs["h_pt"].apply(lambda p: p.x).values
h_y = assign_pairs["h_pt"].apply(lambda p: p.y).values
w_x = assign_pairs["w_pt"].apply(lambda p: p.x).values
w_y = assign_pairs["w_pt"].apply(lambda p: p.y).values

assign_pairs["orig_node"] = ox.distance.nearest_nodes(G_proj, X=h_x, Y=h_y)
assign_pairs["dest_node"] = ox.distance.nearest_nodes(G_proj, X=w_x, Y=w_y)

edge_flow = defaultdict(float)
routed_paths = {}

for idx, row in assign_pairs.iterrows():
    if row["orig_node"] == row["dest_node"]:
        continue
    try:
        path = nx.shortest_path(G_proj, row["orig_node"], row["dest_node"], weight="travel_time")
    except nx.NetworkXNoPath:
        continue
    routed_paths[idx] = path
    for u, v in zip(path[:-1], path[1:]):
        edge_flow[(u, v)] += row["flow"]

print(f"Successfully routed: {len(routed_paths)} of {len(assign_pairs)} pairs")
print(f"Distinct road segments carrying assigned flow: {len(edge_flow)}")


## Step 5: Classify into Low / Medium / High and map it

Only segments that carry *any* assigned commuter flow get a color; everything else stays as thin gray background context. Thresholds are tercile-based (relative to each other), since we have no true road capacity to compare against -- another honest simplification worth naming for readers.

In [ ]:
edges_reset = edges.reset_index()
edges_reset["assigned_flow"] = edges_reset.apply(lambda r: edge_flow.get((r["u"], r["v"]), 0), axis=1)

flowed = edges_reset[edges_reset["assigned_flow"] > 0]
p33, p66 = flowed["assigned_flow"].quantile([0.33, 0.66])

def bucket(v):
    if v <= 0:
        return None
    elif v <= p33:
        return "Low"
    elif v <= p66:
        return "Medium"
    else:
        return "High"

edges_reset["intensity"] = edges_reset["assigned_flow"].apply(bucket)
print(edges_reset["intensity"].value_counts(dropna=False))


In [ ]:
edges_3857 = edges_reset.set_geometry("geometry").set_crs(G_crs).to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 12))

# Background: full network, faint
edges_3857.plot(ax=ax, color="#d9d9d9", linewidth=0.3, zorder=1)

widths = {"Low": 1.0, "Medium": 1.8, "High": 3.0}
for level in ["Low", "Medium", "High"]:
    subset = edges_3857[edges_3857["intensity"] == level]
    subset.plot(ax=ax, color=INTENSITY_COLORS[level], linewidth=widths[level], zorder=2 + ["Low","Medium","High"].index(level))

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=0)

handles = [plt.Line2D([0], [0], color=INTENSITY_COLORS[l], linewidth=3, label=l) for l in ["Low", "Medium", "High"]]
ax.legend(handles=handles, loc="lower right", title="Modeled commuter\nflow intensity", fontsize=9)

ax.set_title("Modeled Commuter Traffic Intensity by Road Segment", fontsize=15, pad=12)
ax.set_axis_off()
fig.text(0.5, 0.02, f"Modeled from top {TOP_N_ASSIGN} commuting flows, routed over the real network -- estimate, not measured volume",
         ha="center", fontsize=8, color="#666")

plt.tight_layout()
plt.savefig("pittsburgh_traffic_intensity_map.png", dpi=200, bbox_inches="tight")
plt.show()


## Step 6: The top 3 commute routes, actually routed

Same idea as the desire lines from Notebook 2, but now following real streets instead of a straight line -- you'll likely see them bend around the rivers and funnel through the same bridges.

In [ ]:
def path_to_linestring(G, path):
    coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in path]
    return LineString(coords)

top3 = assign_pairs.nlargest(3, "flow")

fig, ax = plt.subplots(figsize=(12, 12))
edges_3857.plot(ax=ax, color="#d9d9d9", linewidth=0.3, zorder=1)
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=0)

legend_handles = []
for i, (idx, row) in enumerate(top3.iterrows()):
    if idx not in routed_paths:
        continue
    line = path_to_linestring(G_proj, routed_paths[idx])
    line_gdf = gpd.GeoSeries([line], crs=G_crs).to_crs(epsg=3857)
    line_gdf.plot(ax=ax, color=ROUTE_COLORS[i], linewidth=3.5, zorder=5 + i, solid_capstyle="round")
    label = f"{row['h_tract']} → {row['w_tract']}  ({int(row['flow'])} commuters)"
    legend_handles.append(plt.Line2D([0], [0], color=ROUTE_COLORS[i], linewidth=3, label=label))

ax.legend(handles=legend_handles, loc="lower right", fontsize=8, title="Top 3 routes (home tract → work tract)")
ax.set_title("Top 3 Heaviest Commute Routes", fontsize=15, pad=12)
ax.set_axis_off()

plt.tight_layout()
plt.savefig("pittsburgh_top3_routes_map.png", dpi=200, bbox_inches="tight")
plt.show()


## Step 7: Save exports

In [ ]:
edges_flow_export = edges_reset[["u", "v", "key", "name", "road_class", "length", "assigned_flow", "intensity"]].copy()
edges_flow_export["name"] = edges_flow_export["name"].apply(lambda v: str(v) if isinstance(v, list) else v)
edges_flow_export.to_csv("pittsburgh_road_assigned_flow.csv", index=False)

print("Saved pittsburgh_road_assigned_flow.csv --", len(edges_flow_export), "rows")


In [ ]:
from google.colab import files

files.download("pittsburgh_road_assigned_flow.csv")
files.download("pittsburgh_traffic_intensity_map.png")
files.download("pittsburgh_top3_routes_map.png")


## Recap + what's next

You just:
- Built a green/yellow/red road-level traffic map using real routing over the real network -- a modeled estimate grounded in genuine commuting data, clearly distinguished from measured sensor data
- Routed the three heaviest commutes over actual streets instead of straight lines, and very likely watched them funnel through the same bridges and highway approaches
- Kept the whole pipeline on fully open data -- OSM (ODbL) and Census LODES (public domain) -- no license surprises anywhere in this project

That closes out the EDA half of the series (Projects 1 and 2, seven notebooks total). **Project 3** picks this up as basic modeling -- likely predicting flow or intensity from road/tract features -- ahead of Project 4's live simulation and Project 5's GNN, both of which now have a real, working notion of network-level traffic intensity to build on.